# 🎙️ BookVoice-AI — XTTS-v2 Fine-tuning (Colab)
**v2.5 · Juni 2026**

Trainiert eine eigene Stimme (z.B. türkische Sufi-Stimme) per Fine-tuning auf XTTS-v2.

- **Basis:** `coqui-tts` (idiap-Fork, aktiv gepflegt) — *nicht* das tote Original, *nicht* daswer123.
- **Trainer:** offizieller XTTS-`GPTTrainer` aus genau dieser Library.
- **Dataset:** kommt fertig aus dem BookVoice-Trainingsraum (`audio_file|text|speaker_name`).

**🆕 Neu in v2.5:**
- **RESUME** — von vorhandenem Checkpoint weitertrainieren (z.B. `checkpoint_2000.pth`), statt bei null.
- **Lokale Checkpoints** (Colab-Disk ~78 GB) statt Drive → kein Voll-Laufen mehr.
- **Auto-Löschen** alter Checkpoints (`save_n_checkpoints=1`, `save_all_best=False`) → spart Platz.
- **Produktions-Test** mit den getunten Parametern (Block-Bündelung, Fades, temp 0.65, rep 2.0).

➡️ **Bedienung:** Alle Einstellungen rechts in den **Formularfeldern**. Zellen der Reihe nach mit ▶ ausführen.

**Voraussetzung:** Runtime → *Laufzeittyp ändern* → **T4 GPU**.

> ⚠️ Falls nach Zelle 2 ein CUDA-Fehler kommt: *Runtime → Sitzung neu starten*, dann ab Zelle 1 erneut.


In [ ]:
#@title 1. Server & Projekt einrichten { display-mode: "form" }
import os, requests, zipfile, io, json

SERVER_URL   = "https://ahrar.aksoy-net.de"  #@param {type:"string"}
PROJECT_NAME = ""  #@param {type:"string"}

# API-URL normalisieren (/api wird von nginx auf das TTS-Backend gemappt)
if SERVER_URL.rstrip("/").endswith("/api"):
    API_BASE = SERVER_URL.rstrip("/")
else:
    API_BASE = SERVER_URL.rstrip("/") + "/api"

assert PROJECT_NAME.strip(), "❌ Bitte PROJECT_NAME ausfüllen (rechts im Formular)!"
print(f"🌐 Server : {API_BASE}")
print(f"📁 Projekt: {PROJECT_NAME}")


In [ ]:
#@title 2. coqui-tts installieren (Python 3.12 kompatibel) { display-mode: "form" }
print("⏳ Installiere coqui-tts (idiap-Fork) ...")
# transformers gepinnt: coqui-tts braucht isin_mps_friendly (ab 4.45),
# transformers>=5.1 hat es entfernt -> Bereich 4.45..4.56 ist kompatibel.
!pip install -q coqui-tts "transformers>=4.45,<4.57"
# torchvision rauswerfen: wird fürs TTS nicht gebraucht und verursacht sonst
# "operator torchvision::nms does not exist" beim transformers-Import.
!pip uninstall -q -y torchvision
import torch
print("✅ coqui-tts installiert")
print(f"🔥 PyTorch {torch.__version__}")
print(f"🎮 CUDA: {torch.cuda.is_available()} · {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
if not torch.cuda.is_available():
    print("⚠️  Keine GPU aktiv! Runtime → Laufzeittyp ändern → T4 GPU.")
print("ℹ️  Falls Zelle 6 einen transformers-ImportError zeigt: "
      "Runtime → Sitzung neu starten, dann ab Zelle 1 erneut.")


In [ ]:
#@title 3. Dataset laden { display-mode: "form" }
import shutil
DATASET_QUELLE = "server"  #@param ["server", "lokal_zip"]
EVAL_PROZENT   = 0.15  #@param {type:"slider", min:0.05, max:0.4, step:0.05}
TEXT_CLEANER   = True   #@param {type:"boolean"}
LOKAL_ZIP      = ""     #@param {type:"string"}

DATASET_PATH = "/content/dataset"
if os.path.exists(DATASET_PATH):
    shutil.rmtree(DATASET_PATH)
os.makedirs(DATASET_PATH, exist_ok=True)

if DATASET_QUELLE == "server":
    # 1) Export auf dem Server triggern
    print("⏳ Triggere Export auf dem Server ...")
    exp = requests.post(
        f"{API_BASE}/training/projects/{PROJECT_NAME}/export",
        json={"eval_percentage": EVAL_PROZENT, "apply_cleaner": TEXT_CLEANER},
        timeout=300,
    )
    exp.raise_for_status()
    info = exp.json()
    print(f"✅ Export: {info['train']} Train · {info['eval']} Eval · "
          f"Sprache={info['sprache']} · Speaker={info['speaker_name']}")
    # 2) ZIP herunterladen
    print("⏳ Lade Dataset-ZIP ...")
    dl = requests.get(
        f"{API_BASE}/training/projects/{PROJECT_NAME}/export/download",
        timeout=600,
    )
    dl.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(dl.content)) as z:
        z.extractall(DATASET_PATH)
else:
    # Lokales ZIP vom Dataset-Tool: entweder Pfad (z.B. Drive) oder Upload-Dialog
    if LOKAL_ZIP.strip():
        zpath = LOKAL_ZIP
    else:
        from google.colab import files
        print("⬆️  dataset.zip auswählen:")
        up = files.upload()
        zpath = list(up.keys())[0]
    with zipfile.ZipFile(zpath) as z:
        z.extractall(DATASET_PATH)
    print(f"✅ Lokales Dataset entpackt: {zpath}")

LANGUAGE = open(os.path.join(DATASET_PATH, "lang.txt")).read().strip() or "tr"
n_wav = len([f for f in os.listdir(os.path.join(DATASET_PATH, "wavs")) if f.endswith(".wav")])
print(f"✅ Dataset bereit: {n_wav} WAVs · Sprache: {LANGUAGE}")
print(f"   Pfad: {DATASET_PATH}")


In [ ]:
#@title 4. XTTS-v2 Basis-Checkpoints laden { display-mode: "form" }
from huggingface_hub import hf_hub_download

CKPT_DIR = "/content/xtts_base"
os.makedirs(CKPT_DIR, exist_ok=True)

print("⏳ Lade XTTS-v2 Basis-Dateien von HuggingFace (coqui/XTTS-v2) ...")
paths = {}
for fn in ["dvae.pth", "mel_stats.pth", "vocab.json", "model.pth", "config.json"]:
    paths[fn] = hf_hub_download(repo_id="coqui/XTTS-v2", filename=fn, local_dir=CKPT_DIR)
    print(f"  ✓ {fn}")

DVAE_CKPT  = paths["dvae.pth"]
MEL_STATS  = paths["mel_stats.pth"]
VOCAB      = paths["vocab.json"]
MODEL_CKPT = paths["model.pth"]
print("✅ Basis-Modell bereit")


In [ ]:
#@title 5. Trainings-Parameter { display-mode: "form" }
EPOCHS               = 20    #@param {type:"slider", min:1, max:60, step:1}
BATCH_SIZE           = 3     #@param {type:"slider", min:1, max:8, step:1}
GRAD_ACCUM           = 12    #@param {type:"slider", min:1, max:256, step:1}
LEARNING_RATE        = 5e-06 #@param {type:"number"}
NACH_DRIVE_SPEICHERN = False #@param {type:"boolean"}
RESUME_CHECKPOINT    = ""    #@param {type:"string"}

# Drive immer mounten: noetig, um einen RESUME-Checkpoint von Drive zu LESEN
# + um ihn automatisch zu finden. OUT_PATH bleibt trotzdem lokal (Default) ->
# kein Drive-Voll-Laufen, weil die NEUEN Checkpoints lokal landen.
from google.colab import drive
drive.mount("/content/drive")

if NACH_DRIVE_SPEICHERN:
    OUT_PATH = f"/content/drive/MyDrive/BookVoice_Training/{PROJECT_NAME}"
else:
    OUT_PATH = f"/content/training/{PROJECT_NAME}"
os.makedirs(OUT_PATH, exist_ok=True)

# RESUME: Checkpoint-Pfad-Tipp, falls leer gelassen
if not RESUME_CHECKPOINT.strip():
    import glob as _g
    _cands = sorted(_g.glob("/content/drive/MyDrive/**/checkpoint_*.pth", recursive=True),
                    key=os.path.getmtime)
    # Mini-Dateien rausfiltern: vollstaendiger XTTS-Checkpoint ~5 GB;
    # winzige (<500 MB) sind meist abgebrochene/unvollstaendige Saves -> unbrauchbar.
    _ok  = [c for c in _cands if os.path.getsize(c) > 500*1024*1024]
    _bad = [c for c in _cands if os.path.getsize(c) <= 500*1024*1024]
    if _ok:
        print(f"💡 Tipp: vollständiger Checkpoint zum Fortsetzen: {_ok[-1]}")
        print(f"   ({os.path.getsize(_ok[-1])//1024//1024} MB) → ins Feld RESUME_CHECKPOINT kopieren.")
    for c in _bad:
        print(f"⚠️  Ignoriert (zu klein, wahrscheinlich abgebrochen): {os.path.basename(c)} "
              f"({os.path.getsize(c)//1024//1024} MB)")

eff = BATCH_SIZE * GRAD_ACCUM
print(f"⚙️  Ziel: {EPOCHS} Epochen · batch={BATCH_SIZE} · grad_accum={GRAD_ACCUM} → effektiv {eff}")
print(f"💾 Checkpoints → {OUT_PATH}  (nur je 1 neuester bleibt, aeltere werden geloescht)")
if RESUME_CHECKPOINT.strip():
    assert os.path.exists(RESUME_CHECKPOINT), f"❌ RESUME_CHECKPOINT nicht gefunden: {RESUME_CHECKPOINT}"
    _sz = os.path.getsize(RESUME_CHECKPOINT)//1024//1024
    if _sz < 500:
        print(f"⚠️⚠️  ACHTUNG: RESUME_CHECKPOINT ist nur {_sz} MB — vollständig wären ~5000 MB!")
        print("   Das ist wahrscheinlich ein abgebrochener Save (z.B. checkpoint_2085).")
        print("   → Nimm lieber den großen, vollständigen (z.B. checkpoint_2000, ~5 GB).")
    print(f"↩️  RESUME ab: {RESUME_CHECKPOINT} ({_sz} MB)")
    print("   ⚠️  EPOCHS ist das GESAMT-Ziel inkl. schon trainierter Epochen!")
    print("   ⚠️  z.B. checkpoint_2000 ≈ Epoche 9 → EPOCHS=20 trainiert ~11 weitere.")
else:
    print("🆕 Frischer Lauf (kein RESUME).")


In [ ]:
#@title 6. Training starten { display-mode: "form" }
from trainer import Trainer, TrainerArgs
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig
from TTS.tts.models.xtts import XttsAudioConfig

# --- Custom-Formatter für BookVoice-CSV: audio_file|text|speaker_name (mit Header) ---
def bookvoice_formatter(root_path, meta_file, **kwargs):
    items = []
    with open(os.path.join(root_path, meta_file), "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split("|")
            if len(parts) < 2 or parts[0] == "audio_file":   # Header / leere Zeile überspringen
                continue
            items.append({
                "audio_file": os.path.join(root_path, parts[0]),
                "text": parts[1],
                "speaker_name": parts[2] if len(parts) > 2 else "speaker",
                "language": LANGUAGE,
                "root_path": root_path,
            })
    return items

train_samples = bookvoice_formatter(DATASET_PATH, "metadata_train.csv")
eval_samples  = bookvoice_formatter(DATASET_PATH, "metadata_eval.csv")
print(f"📊 {len(train_samples)} Train · {len(eval_samples)} Eval Samples")
assert train_samples, "❌ Keine Train-Samples — Dataset prüfen."

# --- Modell-Argumente (XTTS-v2 Standard) ---
model_args = GPTArgs(
    max_conditioning_length=132300,   # 6 s
    min_conditioning_length=66150,    # 3 s
    debug_loading_failures=False,
    max_wav_length=255995,            # ~11.6 s
    max_text_length=200,
    mel_norm_file=MEL_STATS,
    dvae_checkpoint=DVAE_CKPT,
    xtts_checkpoint=MODEL_CKPT,       # Basis, auf dem wir aufsetzen
    tokenizer_file=VOCAB,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)
audio_config = XttsAudioConfig(
    sample_rate=22050, output_sample_rate=24000,
)

config = GPTTrainerConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    run_name="BookVoice_XTTS",
    project_name="BookVoice",
    run_description="BookVoice XTTS-v2 Fine-tune",
    dashboard_logger="tensorboard",
    logger_uri=None,
    audio=audio_config,
    batch_size=BATCH_SIZE,
    batch_group_size=48,
    eval_batch_size=BATCH_SIZE,
    num_loader_workers=2,
    eval_split_max_size=256,
    print_step=50,
    plot_step=100,
    log_model_step=500,
    save_step=500,
    save_n_checkpoints=1,
    save_checkpoints=True,
    save_all_best=False,
    save_best_after=0,
    print_eval=False,
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=LEARNING_RATE,
    lr_scheduler="MultiStepLR",
    lr_scheduler_params={"milestones": [900000, 2700000, 5400000], "gamma": 0.5, "last_epoch": -1},
    epochs=EPOCHS,
    test_sentences=[],
)

model = GPTTrainer.init_from_config(config)

trainer = Trainer(
    TrainerArgs(
        restore_path=RESUME_CHECKPOINT or None,
        skip_train_epoch=False,
        start_with_eval=False,
        grad_accum_steps=GRAD_ACCUM,
    ),
    config,
    output_path=OUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

print("🚀 Training läuft ... Checkpoints landen in OUT_PATH (nur je 1 neuester bleibt).")
print("   (Bei Disconnect: RESUME_CHECKPOINT in Zelle 5 auf den letzten checkpoint_*.pth")
print("    setzen und ab Zelle 5 weiter.)")
trainer.fit()
print("🎉 Training fertig! Bestes Modell (best_model.pth) liegt im OUT_PATH.")


In [ ]:
#@title 7. Stimme testen — Produktions-Pipeline { display-mode: "form" }
import glob, re, numpy as np, soundfile as sf
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from IPython.display import Audio, display

# ── Test-Text hier reinschreiben (mehrzeilig erlaubt) ──────────────
TEST_TEXT = """Bugün hava çok güzel ve güneşli. Sabah erkenden kalktım, kahvaltımı yaptım.
İnsan hayatında en önemli şey sevgi ve saygıdır. Bilgi insanı özgür kılar.
Zaman su gibi akıp gidiyor, her an değerlidir. Doğru yolda yürümek cesaret ister.
Kitap okumak zihni besler ve dünyayı genişletir."""
# ───────────────────────────────────────────────────────────────────

TEMPERATURE = 0.65 #@param {type:"slider", min:0.3, max:1.0, step:0.05}
REP_PENALTY = 2.0  #@param {type:"slider", min:1.0, max:10.0, step:0.5}
SPEED       = 1.0  #@param {type:"slider", min:0.8, max:1.2, step:0.05}
PAUSE_MS    = 280  #@param {type:"slider", min:0, max:800, step:20}
BLOCK_LEN   = 240  #@param {type:"slider", min:80, max:250, step:10}
N_REF       = 15   #@param {type:"slider", min:1, max:30, step:1}

# jüngstes trainiertes Modell + config finden
runs = sorted(
    glob.glob(os.path.join(OUT_PATH, "*", "best_model*.pth")) +
    glob.glob(os.path.join(OUT_PATH, "*", "checkpoint_*.pth")),
    key=os.path.getmtime,
)
assert runs, "❌ Kein trainiertes Modell gefunden — erst Zelle 6 laufen lassen."
TRAINED_CKPT = runs[-1]
TRAINED_DIR  = os.path.dirname(TRAINED_CKPT)
TRAINED_CFG  = os.path.join(TRAINED_DIR, "config.json")
print(f"📦 Modell: {TRAINED_CKPT}")

# Referenz: die N GRÖSSTEN (= längsten) Clips -> stabiles Embedding
allw = glob.glob(os.path.join(DATASET_PATH, "wavs", "*.wav"))
allw.sort(key=lambda p: os.path.getsize(p), reverse=True)
SPEAKER_REF = allw[:N_REF]
print(f"🎚️  Referenz: {len(SPEAKER_REF)} größte Clips")

cfg = XttsConfig(); cfg.load_json(TRAINED_CFG)
xtts = Xtts.init_from_config(cfg)
try:
    xtts.load_checkpoint(cfg, checkpoint_path=TRAINED_CKPT, vocab_path=VOCAB, use_deepspeed=False)
except Exception as e:
    import torch; print("ℹ️  Fallback model-State ...", str(e)[:90])
    ck = torch.load(TRAINED_CKPT, map_location="cpu"); st = ck.get("model", ck)
    torch.save(st, "/content/_m.pth")
    xtts.load_checkpoint(cfg, checkpoint_path="/content/_m.pth", vocab_path=VOCAB, use_deepspeed=False)
xtts.cuda()

# Conditioning-Latents EINMAL aus vielen Clips -> konsistente, stabile Stimme
gpt_latent, speaker_emb = xtts.get_conditioning_latents(
    audio_path=SPEAKER_REF, gpt_cond_len=30, max_ref_length=60)

# Sätze in Blöcke bündeln (bis BLOCK_LEN Zeichen) -> Betonung fließt über Sätze
def in_bloecke(text, max_len=240):
    saetze = [s.strip() for s in re.split(r"(?<=[.!?;…])\s+", text.replace("\n", " ").strip()) if s.strip()]
    out, cur = [], ""
    for s in saetze:
        if len(cur) + len(s) + 1 <= max_len:
            cur = (cur + " " + s).strip()
        else:
            if cur: out.append(cur)
            cur = s
    if cur: out.append(cur)
    return out

# kleine Kanten-Fades gegen Klicks an den Übergängen
def fade(a, n=200):
    a = a.copy()
    if len(a) > 2*n:
        a[:n]  *= np.linspace(0, 1, n, dtype=np.float32)
        a[-n:] *= np.linspace(1, 0, n, dtype=np.float32)
    return a

bloecke = in_bloecke(TEST_TEXT, BLOCK_LEN)
print(f"📝 {len(bloecke)} Blöcke")

pause = np.zeros(int(24000 * PAUSE_MS / 1000), dtype=np.float32)
chunks = []
for i, b in enumerate(bloecke, 1):
    res = xtts.inference(
        text=b, language=LANGUAGE,
        gpt_cond_latent=gpt_latent, speaker_embedding=speaker_emb,
        temperature=TEMPERATURE, length_penalty=1.0, repetition_penalty=REP_PENALTY,
        top_k=50, top_p=0.85, speed=SPEED,
    )
    chunks.append(fade(np.asarray(res["wav"], dtype=np.float32)))
    chunks.append(pause)
    print(f"  ✓ {i}/{len(bloecke)}")

audio = np.concatenate(chunks)
audio = audio / (np.max(np.abs(audio)) + 1e-8) * 0.95   # sanfte Gesamt-Normalisierung
sf.write("/content/test_output.wav", audio, 24000)
print(f"🔊 Fertig · {len(audio)/24000:.1f}s")
display(Audio("/content/test_output.wav", rate=24000))


In [ ]:
#@title 8. Modell zurück an BookVoice hochladen { display-mode: "form" }
import shutil
MODELL_NAME = "sufi_xtts"  #@param {type:"string"}

# Modell-Paket bauen: model.pth + config.json + vocab.json (Rest ergänzt der Server)
pack_dir = "/content/model_pack"
if os.path.exists(pack_dir):
    shutil.rmtree(pack_dir)
os.makedirs(pack_dir, exist_ok=True)
shutil.copy2(TRAINED_CKPT, os.path.join(pack_dir, "model.pth"))
shutil.copy2(TRAINED_CFG,  os.path.join(pack_dir, "config.json"))
shutil.copy2(VOCAB,        os.path.join(pack_dir, "vocab.json"))

zip_base = f"/content/{MODELL_NAME}"
shutil.make_archive(zip_base, "zip", pack_dir)
zip_path = zip_base + ".zip"
print(f"📦 ZIP: {zip_path} ({os.path.getsize(zip_path)//1024//1024} MB)")

print("⏳ Lade zu BookVoice hoch ...")
with open(zip_path, "rb") as f:
    up = requests.post(
        f"{API_BASE}/training/models/upload",
        data={"name": MODELL_NAME},
        files={"file": (f"{MODELL_NAME}.zip", f, "application/zip")},
        timeout=600,
    )
if up.ok:
    print("✅ Hochgeladen!", up.json())
    print("   → In BookVoice: Trainingsraum → Modelle → aktivieren.")
else:
    print(f"❌ Fehler {up.status_code}: {up.text[:300]}")
